# ML-03 — Frame Your Lane as an ML Task

This notebook frames a content-refresh priority queue before any modelling. The goal is a better editorial decision, not a model for its own sake.

## 1. My lane as an ML task (type)

I am framing this as a **ranking / scoring** task. A content editor needs to decide which pages to review for a possible refresh first. The output would be a priority score for each content item, then a ranked queue. This is not classification because the useful action is to choose an ordered batch, rather than make a yes/no decision about every page.

The actor is a content editor. They would investigate the 20 highest-priority pages and decide whether each needs an update, rewrite, or no action. A wrong high ranking costs editor time; a wrong low ranking can leave a valuable declining page unreviewed. This notebook makes only a decision-support framing claim.

In [1]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path('../../data/raw/content_refresh_anonymized.csv')

## 2. Target or proxy

The eventual target should be an **observed future outcome**, for example `refresh_worthwhile_observed`: 1 when a page that is reviewed and refreshed later shows a pre-defined improvement in organic performance, and 0 otherwise. The starter snapshot does not contain that future outcome, so I cannot honestly train or evaluate it yet.

For this framing exercise only, I sketch a temporary `decline_proxy_rule` from the existing `trend_direction` field. It indicates whether the page was labelled `down` from its recent impressions trend. This is a defined rule, not an observed target. Because `trend_direction` and `trend_pct` create the proxy, they must never be features in a later model.

In [2]:
target_sketch = pd.DataFrame(
    {
        'column': ['refresh_worthwhile_observed', 'decline_proxy_rule'],
        'meaning': [
            'Future observed outcome after review/refresh; the eventual target.',
            'Temporary rule-based sketch: trend_direction equals down.'
        ],
        'available_in_starter_snapshot': [False, True],
        'safe_as_a_future_model_feature': [False, False],
    }
)
target_sketch

,column,meaning,available_in_starter_snapshot,safe_as_a_future_model_feature
0,refresh_worthwhile_observed,Future observed outcome after review/refresh; ...,False,False
1,decline_proxy_rule,Temporary rule-based sketch: trend_direction e...,True,False


## 3. Success metric

I will use **Precision@20**. It asks: of the first 20 pages in the ranked queue, what proportion are genuinely worthwhile refresh candidates according to the eventual observed target? For example, if 15 of the top 20 prove worthwhile, Precision@20 is 15 / 20 = 0.75.

This fits the action because 20 pages is a manageable review batch for an editor. The metric cannot be calculated honestly from the starter snapshot yet, because its observed future target is not available.

In [3]:
metric_definition = pd.DataFrame(
    [{
        'metric': 'Precision@20',
        'queue_size': 20,
        'calculation': 'worthwhile pages among the top 20 / 20',
        'editor_action': 'Investigate the top 20 pages first',
    }]
)
metric_definition

,metric,queue_size,calculation,editor_action
0,Precision@20,20,worthwhile pages among the top 20 / 20,Investigate the top 20 pages first


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content item (page) over the trailing 90-day measurement window.** I load the 30,000-row starter dataset and keep columns an editor could use to understand a candidate. I do not display identifiers, client names, URLs, or queries. Rates are stored as percentages: for example, `ctr = 0.76` means 0.76%, not 76%.

In [4]:
content = pd.read_csv(DATA_PATH)

# Temporary proxy only: it is derived from trend_direction and is not a model feature.
content['decline_proxy_rule'] = content['trend_direction'].eq('down')

lane_columns = [
    'content_type',
    'main_intent',
    'content_age_days',
    'days_since_last_update',
    'impressions_90d',
    'clicks_90d',
    'ctr',
    'avg_position',
    'engagement_rate',
    'decline_proxy_rule',
]
lane_df = content.loc[:, lane_columns]

print(f'Rows in the content-refresh lane: {len(lane_df):,}')
print(f'Unit of analysis: one row = one content item (page)')
lane_df.head()

Rows in the content-refresh lane: 30,000
Unit of analysis: one row = one content item (page)


,content_type,main_intent,content_age_days,days_since_last_update,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,decline_proxy_rule
0,keyword article,transactional,187,20,3803,29,0.76,10.6,5.88,True
1,keyword article,informational,445,25,15320,7,0.05,20.3,0.00,True
2,keyword article,informational,141,20,12581,11,0.09,36.5,0.00,True
3,keyword article,commercial,463,22,11751,58,0.49,6.2,1.28,False
4,keyword article,informational,263,14,19140,24,0.13,44.0,0.00,True


## 5. Why ML beats a fixed rule here

A fixed rule such as “review every page with low CTR” is too simple. A low CTR can mean a poor title, an irrelevant search query, a weak ranking position, or a page that gets very little traffic. A high-priority refresh candidate may instead combine meaningful impressions, worsening performance, stale content, weak engagement, and an achievable ranking position. These signals interact and their useful thresholds can differ across content types and client contexts.

A future scoring model could learn a consistent priority pattern from observed outcomes and still leave the editor in control of the final refresh decision. Before claiming that it improves on a rule, I would compare its Precision@20 to a transparent baseline and validate on held-out clients or future time periods.

In [5]:
signal_summary = pd.DataFrame(
    [
        ('Search opportunity', 'impressions_90d, ctr, avg_position'),
        ('Content freshness', 'content_age_days, days_since_last_update'),
        ('Audience response', 'clicks_90d, engagement_rate'),
    ],
    columns=['signal family', 'example available fields'],
)
signal_summary

,signal family,example available fields
0,Search opportunity,"impressions_90d, ctr, avg_position"
1,Content freshness,"content_age_days, days_since_last_update"
2,Audience response,"clicks_90d, engagement_rate"


## Self-check

- [x] I named the task type: ranking/scoring for a content-refresh queue.
- [x] I named an honest future target and labelled the available decline field as only a rule-based proxy.
- [x] I chose Precision@20 and tied it to an editor reviewing 20 pages.
- [x] I showed the real unit of analysis in a dataframe: one row per content item.
- [x] I explained why interacting signals can beat a single fixed rule.
- [x] The notebook runs top to bottom without errors.
- [ ] I will commit this executed notebook under `work/notebooks/` and submit my repo URL.